<a href="https://colab.research.google.com/github/craljimenez/An-lisis_y_Visualizacion_de_Datos_para_la_Toma_de_Decisiones/blob/main/clases/semana03-clase05-bases-datos-sql-basico/notebooks/lab-sql-basico.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Laboratorio: Bases de datos relacionales y SQL básico
### Semana 3 · Clase 5
**Curso:** Análisis y Visualización de Datos para la Toma de Decisiones (Esumer)

En la Clase 4 le hicimos preguntas de negocio a un `DataFrame` de pandas: filtrar filas, elegir columnas, ordenar, resumir. Hoy hacemos exactamente las mismas preguntas, pero sobre datos que viven en una **base de datos relacional**, con el lenguaje hecho para eso: **SQL**.

Vamos a trabajar con `tienda.db`: una base de datos [SQLite](https://www.sqlite.org/) de una tienda en línea ficticia ("Tienda Andina"), con 5 tablas relacionadas: `clientes`, `empleados`, `productos`, `pedidos` y `detalle_pedido`.

SQLite tiene una ventaja enorme para aprender: **no necesita instalar ni configurar ningún servidor**. Toda la base de datos vive en un solo archivo (`tienda.db`), y Python ya trae integrado todo lo necesario para leerlo (el módulo `sqlite3`, parte de la librería estándar — no hay que instalar nada).

## 1. Subir `tienda.db` a Google Colab

Primero hay que llevar el archivo desde tu computador hasta el entorno de Colab.

Antes de continuar, descarga `tienda.db` a tu computador desde la carpeta `datos/` de esta misma clase, en el repositorio del curso. Después ejecuta la celda siguiente y selecciónalo.

In [ ]:
from google.colab import files

uploaded = files.upload()

Recuerda que el entorno de Colab es **efímero** (lo vimos en la Clase 4): si cierras la pestaña o reinicias el entorno, `tienda.db` se borra y hay que volver a subirlo. Eso no afecta al notebook en sí (si lo guardas en tu Drive), solo al archivo de datos de esa sesión.

## 2. Conectarse a la base de datos

Para trabajar con SQLite en Python se usa el módulo `sqlite3` (viene incluido con Python, no se instala aparte). `sqlite3.connect()` abre una **conexión** al archivo `.db` — el mismo concepto de "conexión" que usarías para conectarte a cualquier otro motor de base de datos (PostgreSQL, MySQL, SQL Server...), solo que aquí no hace falta usuario, contraseña ni dirección de servidor: todo vive en el archivo.

In [ ]:
import sqlite3
import pandas as pd

con = sqlite3.connect("tienda.db")
print("Conexión abierta correctamente.")

Para ejecutar una consulta SQL y ver el resultado como una tabla, usamos `pandas.read_sql_query(consulta, conexion)`: le pasas el texto de la consulta y la conexión, y te devuelve un `DataFrame` — la misma estructura que ya conoces de la Clase 4. Esta es la manera más cómoda de combinar SQL con todo lo que ya sabes de pandas.

In [ ]:
pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table';", con)

Esa consulta usa una tabla especial que SQLite mantiene automáticamente (`sqlite_master`) para listar todo lo que hay en la base de datos. Confirmamos así las 5 tablas: `clientes`, `empleados`, `productos`, `pedidos` y `detalle_pedido` — las mismas del esquema relacional que vimos en las diapositivas.

## 3. Tabla, registro y campo: explorando la estructura

Antes de consultar datos, conviene mirar la **estructura** de una tabla: qué campos (columnas) tiene y de qué tipo es cada uno. `PRAGMA table_info(tabla)` es la forma de preguntárselo a SQLite (equivalente a `df.dtypes` o `df.info()` en pandas).

### Esquema relacional de `tienda.db`

Ten esta imagen a la mano durante todo el laboratorio: es el mapa de las 5 tablas y cómo se conectan (clave primaria de cada tabla y las claves foráneas que las relacionan). Te va a servir para saber en qué tabla está cada dato y cómo se llama cada columna, sin tener que adivinar.

![Esquema relacional de tienda.db](https://raw.githubusercontent.com/craljimenez/An-lisis_y_Visualizacion_de_Datos_para_la_Toma_de_Decisiones/main/clases/semana03-clase05-bases-datos-sql-basico/figuras/figura_esquema_relacional.png)

*(Es una versión simplificada: cada tabla tiene además un par de columnas más, que vas a ir descubriendo con `PRAGMA table_info()` en la siguiente celda.)*

In [ ]:
pd.read_sql_query("PRAGMA table_info(clientes);", con)

La columna `pk` marca cuál campo es la **clave primaria** (`1` = sí, `0` = no): en `clientes`, es `id_cliente`. La columna `notnull` marca si el campo puede quedar vacío (`1` = obligatorio).

Actividad: usa `PRAGMA table_info()` para explorar la estructura de la tabla `pedidos`. ¿Cuál es su clave primaria? ¿Qué campos reconocerías como claves foráneas (los que apuntan a otra tabla), según el esquema relacional de las diapositivas?

In [ ]:
# TODO: usa PRAGMA table_info() sobre la tabla "pedidos"


## 4. La consulta más simple: SELECT y FROM

`SELECT` dice qué columnas quieres ver; `FROM` dice de qué tabla. Empecemos por traer una tabla completa.

In [ ]:
pd.read_sql_query("SELECT * FROM clientes;", con)

`*` significa "todas las columnas". Pero, igual que discutimos en las diapositivas, casi siempre es mejor **nombrar las columnas que sí necesitas**: es más claro y más seguro (si alguien agrega una columna nueva a la tabla, tu consulta no cambia de resultado sin que tú lo hayas pedido).

In [ ]:
pd.read_sql_query("SELECT nombre, ciudad, segmento FROM clientes;", con)

## 5. LIMIT: explorar sin traer todo

Cuando solo quieres ver "cómo se ve" una tabla, sin traerla completa, `LIMIT` corta el resultado a las primeras *n* filas (el equivalente de `df.head()` en pandas).

In [ ]:
pd.read_sql_query("SELECT nombre, precio, stock FROM productos LIMIT 5;", con)

## 6. WHERE: filtrar filas

`WHERE` se queda solo con las filas que cumplen una condición — la misma idea que `df[condición]` en pandas.

In [ ]:
pd.read_sql_query("""
SELECT nombre, ciudad, segmento
FROM clientes
WHERE ciudad = 'Medellín';
""", con)

Nota la sintaxis de Python: como la consulta SQL tiene varias líneas, la escribimos entre **triples comillas** (`"""`), que en Python permiten texto de varias líneas. El texto de comillas simples dentro (`'Medellín'`) es parte del SQL, no de Python.

Actividad: escribe una consulta que traiga `nombre` y `precio` de la tabla `productos`, solo para la categoría `'Papelería'`.

In [ ]:
# TODO: SELECT nombre, precio FROM productos WHERE categoria = 'Papelería'


## 7. Operadores lógicos: AND, OR, NOT — y el cuidado con los paréntesis

Recuerda del contenido de hoy: SQL evalúa `AND` **antes** que `OR`. Vamos a comprobarlo con los mismos datos, comparando el resultado con y sin paréntesis.

In [ ]:
sin_parentesis = pd.read_sql_query("""
SELECT nombre, categoria, precio
FROM productos
WHERE categoria = 'Hogar' OR categoria = 'Oficina' AND precio > 500000;
""", con)
print(f"Sin paréntesis: {len(sin_parentesis)} filas")
sin_parentesis

In [ ]:
con_parentesis = pd.read_sql_query("""
SELECT nombre, categoria, precio
FROM productos
WHERE (categoria = 'Hogar' OR categoria = 'Oficina') AND precio > 500000;
""", con)
print(f"Con paréntesis: {len(con_parentesis)} filas")
con_parentesis

Con paréntesis, la consulta trae **solo** productos de Hogar u Oficina que además cuesten más de \$500.000. Sin paréntesis, `AND` se evalúa primero: el resultado es "todo Hogar" (sin importar el precio) más "Oficina caro" — mucho menos restrictivo de lo que probablemente querías. **Cuando combines `AND` y `OR`, usa paréntesis siempre**, aunque creas que no los necesitas.

## 8. IN, BETWEEN, LIKE e IS NULL

Cuatro atajos muy comunes para condiciones que, escritas solo con `=`, `AND` y `OR`, quedarían muy largas.

**IN** — pertenece a una lista de valores:

In [ ]:
pd.read_sql_query("""
SELECT nombre, ciudad
FROM clientes
WHERE ciudad IN ('Cali', 'Bogotá', 'Barranquilla');
""", con)

**BETWEEN** — dentro de un rango (incluye los dos extremos):

In [ ]:
pd.read_sql_query("""
SELECT id_pedido, fecha, estado
FROM pedidos
WHERE fecha BETWEEN '2026-03-01' AND '2026-03-31';
""", con)

**LIKE** — coincide con un patrón de texto (`%` reemplaza cualquier cantidad de caracteres):

In [ ]:
pd.read_sql_query("""
SELECT nombre, categoria, precio
FROM productos
WHERE nombre LIKE '%portátil%';
""", con)

**IS NULL** — el campo quedó vacío (recuerda: nunca se usa `= NULL`, porque nada es *igual* a un valor desconocido):

In [ ]:
pd.read_sql_query("""
SELECT nombre, ciudad
FROM clientes
WHERE correo IS NULL;
""", con)

Actividad: escribe una consulta que traiga `nombre` y `precio` de `productos` cuyo precio esté entre `100000` y `300000` (usa `BETWEEN`).

In [ ]:
# TODO: SELECT nombre, precio FROM productos WHERE precio BETWEEN 100000 AND 300000


## 9. ORDER BY: ordenar resultados

Por defecto ordena ascendente (`ASC`); para descendente se agrega `DESC`.

In [ ]:
pd.read_sql_query("""
SELECT nombre, precio
FROM productos
ORDER BY precio DESC;
""", con)

## 10. Todo junto: WHERE + ORDER BY + LIMIT

Pregunta de negocio: *¿cuáles son los 3 productos con menos stock disponible?*

In [ ]:
pd.read_sql_query("""
SELECT nombre, stock
FROM productos
WHERE stock < 15
ORDER BY stock ASC
LIMIT 3;
""", con)

Filtrar, ordenar y limitar se combinan todo el tiempo, en ese mismo orden de escritura: `SELECT` ... `FROM` ... `WHERE` ... `ORDER BY` ... `LIMIT`.

## 11. Pensamiento tabular: lo mismo, con otro lenguaje

La tabla de la Clase 4 comparaba pandas con la idea general de SQL. Ahora que ya conoces las cláusulas de verdad, esta es la comparación completa:

| Lo que hicimos en pandas (Clase 4) | Su equivalente en SQL (hoy) |
|---|---|
| `df[["producto", "total"]]` | `SELECT producto, total` |
| `df[df["ciudad"] == "Medellin"]` | `WHERE ciudad = 'Medellin'` |
| `df[(cond1) & (cond2)]` | `WHERE cond1 AND cond2` |
| `df[(cond1) \| (cond2)]` | `WHERE cond1 OR cond2` |
| `df["ciudad"].isin([...])` | `WHERE ciudad IN (...)` |
| `df["correo"].isna()` | `WHERE correo IS NULL` |
| `df.sort_values("precio", ascending=False)` | `ORDER BY precio DESC` |
| `df.head(5)` | `LIMIT 5` |

La idea de fondo es siempre la misma: partes de una tabla completa, le haces una pregunta, y el resultado es otra tabla — solo cambia el lenguaje con el que la escribes.

## 12. Reto: responder preguntas de negocio con SQL

Usa `pd.read_sql_query()` sobre `con` para responder cada pregunta. Cada consulta debe usar solo lo visto hoy: `SELECT`, `FROM`, `WHERE`, `ORDER BY`, `LIMIT` y los operadores de comparación/lógicos.

**Pregunta 1:** ¿cuáles son los 5 clientes que se registraron más recientemente (`fecha_registro`)?

In [ ]:
# TODO: responde la pregunta 1


**Pregunta 2:** ¿cuántos clientes son del segmento `'Empresa'` y viven en `'Medellín'` o `'Bogotá'`? (pista: `IN`, y cuenta las filas del resultado con `len(...)`)

In [ ]:
# TODO: responde la pregunta 2


**Pregunta 3:** ¿qué productos cuestan entre \$100.000 y \$300.000, ordenados del más barato al más caro?

In [ ]:
# TODO: responde la pregunta 3


**Pregunta 4:** ¿qué clientes tienen `'Restrepo'` en su nombre (como primer nombre, apellido, donde sea)? (pista: `LIKE '%Restrepo%'`)

In [ ]:
# TODO: responde la pregunta 4


**Pregunta 5 (reto extra):** ¿qué clientes no tienen correo registrado **y además** viven en `'Medellín'` o `'Envigado'`? (pista: vas a necesitar `IS NULL`, `OR` **y** paréntesis para que quede bien)

In [ ]:
# TODO: responde la pregunta 5 (reto extra)


## Cierre

¿Qué se pareció más a lo que ya sabías de pandas, y qué se sintió más distinto? Guarda una copia de este notebook en tu Google Drive antes de cerrar Colab.

En la **Clase 6 (MasterClass)** recapitulamos todo el Módulo 1 — estadística descriptiva, Python y SQL — antes de entrar de lleno al Módulo 2. En la **Clase 7** retomamos pandas para diagnóstico de calidad de datos: vas a reconocer `IS NULL` de hoy en la idea de "valores nulos" que trabajaremos allá.

## Apéndice: usar el mismo archivo en DBeaver (opcional)

Si prefieres una herramienta visual de bases de datos en vez de escribir SQL desde Python, puedes abrir exactamente el mismo `tienda.db` en [DBeaver](https://dbeaver.io/) (gratuito), sin necesidad de ningún servidor:

1. Descarga e instala DBeaver Community Edition.
2. **Nueva conexión** (ícono del enchufe, o `Archivo → Nueva conexión`) → elige **SQLite**.
3. En "Ruta del archivo de base de datos", selecciona el `tienda.db` que ya descargaste a tu computador (el mismo archivo, no hace falta subirlo a ningún lado).
4. Conecta. En el panel de la izquierda vas a ver las 5 tablas (`clientes`, `empleados`, `productos`, `pedidos`, `detalle_pedido`) — el mismo esquema relacional de las diapositivas.
5. Abre un **editor SQL** (`SQL Editor → New SQL script`) y ejecuta cualquiera de las consultas de este notebook tal cual, con el botón de ejecutar (▶) o `Ctrl+Enter`.

Es la misma base de datos y el mismo SQL en los dos caminos — la diferencia es solo la herramienta desde la que lo escribes. Usa el que te resulte más cómodo para el reto de hoy.